# الفصل 26: دمج النماذج: Model Merging

> تخيّل أنك قمت بضبط نفس النموذج الأساسي ثلاث مرات منفصلة: مرة لتحسين الترميز، ومرة لتحسين المحادثة العامة، ومرة لتحسين الرياضيات.

- **الجزء:** 6 — إعادة استخدام النماذج
- **الفصل:** 26
- **المستوى:** متقدم
- **زمن القراءة:** 20 دقيقة
- **المتطلبات:** الفصل 25: تقطير المعرفة: Knowledge Distillation
- **الوسوم:** Model Merging، SLERP، TIES، DARE، Task Arithmetic

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch26-model-merging.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-06-model-reuse%2Fch26-model-merging.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-06-model-reuse/ch26-model-merging.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch26-model-merging.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch26-model-merging.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch26-model-merging.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch26-model-merging.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-06-model-reuse%2Fch26-model-merging.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-06-model-reuse/ch26-model-merging.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch26-model-merging.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch26-model-merging.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch26-model-merging.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> تخيّل أنك قمت بضبط نفس النموذج الأساسي ثلاث مرات منفصلة: مرة لتحسين الترميز، ومرة لتحسين المحادثة العامة، ومرة لتحسين الرياضيات.

> **بيانات الفصل**
>
> - **الجزء:** 6 — إعادة استخدام النماذج
> - **المستوى:** متقدم
> - **زمن القراءة:** 20 دقيقة
> - **المتطلبات:** الفصل 25: تقطير المعرفة: Knowledge Distillation


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم Model Soup: المتوسط البسيط الذي ينجح بشكل مُدهش
- فهم Task Arithmetic وTask Vectors
- فهم TIES-Merging وDARE: حل مشكلة تداخل التحديثات


## 1. لماذا يهمّك هذا الموضوع؟

🟢 طبقة الحدس
تخيّل أنك قمت بضبط نفس النموذج الأساسي ثلاث مرات منفصلة: مرة لتحسين الترميز، ومرة لتحسين المحادثة العامة، ومرة لتحسين الرياضيات. بدل اختيار واحد منها فقط أو تدريب نموذج رابع يجمعها (مكلف)، ماذا لو استطعت ببساطة "دمج" أوزان الثلاثة نماذج رياضيًا (بدون أي تدريب إضافي!) للحصول على نموذج واحد يمتلك مزايا الثلاثة معًا؟ هذا ممكن فعليًا، وهو أحد أكثر الاكتشافات العملية إثارة للدهشة في 2023-2024 — يُفسَّر غالبًا بأن النماذج المُضبَّطة من نفس الأساس تبقى في "منطقة ملساء ومتصلة" من فضاء الخسارة (Loss Landscape).


## 2. الخريطة الذهنية


```text
فضاء دمج النماذج
│
├── 1) دمج الأوزان (Weight Merging)
│   ├── Model Soup            متوسط حسابي/جشع لنسخ من نفس المهمة
│   ├── Task Arithmetic       جمع/طرح "متجهات المهام" (Task Vectors)
│   ├── TIES-Merging          تقليم + انتخاب الإشارة + دمج منفصل
│   ├── DARE                  إسقاط عشوائي + إعادة قياس قبل الدمج
│   └── SLERP                 استيفاء كروي بين نموذجين فقط
│
└── 2) دمج البنية (Architecture Merging)
    └── Frankenmerging        تركيب طبقات كاملة من نماذج مختلفة
```


الفكرة المركزية: كل تقنيات القسم الأول تعمل على **نفس المعمارية** وتفترض أن
النماذج المُدمَجة تشتركت في تدريب مسبق واحد. أما القسم الثاني فيغيّر البنية
نفسها (عدد الطبقات أو ترتيبها)، وهو أقرب إلى «تجميع» من «مزج».


## 3. الشرح المتدرج

### 3.1 Model Soup: المتوسط البسيط الذي ينجح بشكل مُدهش

🟡 طبقة الآلية

أبسط أشكال الدمج: اجمع عدة نسخ من نفس المعمارية (مُدرَّبة بإعدادات/بذور عشوائية مختلفة قليلاً) بأخذ المتوسط الحسابي البسيط لأوزانها.


In [ ]:
import torch
import copy

def uniform_model_soup(model_checkpoints: list) -> dict:
    """
    Model Soup بأبسط صوره (Wortsman et al., 2022): متوسط حسابي بسيط
    لأوزان عدة نماذج بنفس المعمارية تمامًا - عادة نسخ من نفس النموذج
    مُدرَّبة بـ hyperparameters مختلفة قليلاً (بذور عشوائية، LR، إلخ).
    النتيجة غالبًا أفضل من أي نموذج فردي، وأحيانًا أفضل من أفضل نموذج
    في المجموعة - بدون أي تكلفة تدريب إضافية!
    """
    averaged_state_dict = copy.deepcopy(model_checkpoints[0])
    n_models = len(model_checkpoints)

    for key in averaged_state_dict.keys():
        # جمع نفس الوزن عبر كل النماذج ثم القسمة على العدد
        stacked = torch.stack([ckpt[key].float() for ckpt in model_checkpoints])
        averaged_state_dict[key] = stacked.mean(dim=0).to(model_checkpoints[0][key].dtype)

    return averaged_state_dict

def greedy_model_soup(model_checkpoints: list, validation_fn, model_template) -> dict:
    """
    Greedy Soup (أكثر فعالية من Uniform Soup): ترتيب النماذج حسب أدائها
    الفردي على مجموعة تحقق، ثم إضافتها واحدًا تلو الآخر للمزيج - مع
    الاحتفاظ فقط بالإضافات التي تُحسِّن (أو لا تُضعِف) أداء المزيج الحالي.
    """
    # الخطوة 1: تقييم كل نموذج منفردًا وترتيبها تنازليًا
    scored_models = []
    for ckpt in model_checkpoints:
        model_template.load_state_dict(ckpt)
        score = validation_fn(model_template)
        scored_models.append((ckpt, score))
    scored_models.sort(key=lambda x: x[1], reverse=True)

    # الخطوة 2: البدء بأفضل نموذج، ثم إضافة الباقي بالتتابع
    current_soup = copy.deepcopy(scored_models[0][0])
    model_template.load_state_dict(current_soup)
    current_best_score = validation_fn(model_template)
    n_ingredients = 1

    for ckpt, _ in scored_models[1:]:
        # تجربة إضافة هذا النموذج للمزيج الحالي (متوسط مُرجَّح بعدد المكوّنات)
        candidate_soup = {}
        for key in current_soup.keys():
            candidate_soup[key] = (
                current_soup[key].float() * n_ingredients + ckpt[key].float()
            ) / (n_ingredients + 1)
            candidate_soup[key] = candidate_soup[key].to(current_soup[key].dtype)

        model_template.load_state_dict(candidate_soup)
        candidate_score = validation_fn(model_template)

        if candidate_score >= current_best_score:
            current_soup = candidate_soup
            current_best_score = candidate_score
            n_ingredients += 1
            print(f"✅ تمت إضافة نموذج - الدرجة الجديدة: {candidate_score:.4f}")
        else:
            print(f"❌ تم تجاهل نموذج - لم يُحسِّن الأداء ({candidate_score:.4f})")

    return current_soup


### 3.2 Task Arithmetic وTask Vectors

🟡 طبقة الآلية

فكرة أعمق وأكثر مرونة من Hinton Soup (التي تتطلب نفس البيانات/المهمة تقريبًا): Task Vectors (Ilharco et al., 2022) تسمح بدمج نماذج مُضبَّطة لمهام مختلفة تمامًا.


```text
Task Vector لمهمة معينة = أوزان_النموذج_المُضبَّط - أوزان_النموذج_الأساسي

τ_task = θ_finetuned - θ_base

لإضافة مهارة لنموذج: θ_new = θ_base + λ × τ_task
لإزالة مهارة من نموذج (Forgetting via Negation): θ_new = θ_base - λ × τ_task
لدمج عدة مهارات معًا: θ_new = θ_base + λ₁×τ_1 + λ₂×τ_2 + ... + λₙ×τₙ
```


In [ ]:
class TaskArithmetic:
    """
    تطبيق Task Vectors (Ilharco et al., 2022): استخراج ودمج "متجهات
    المهام" للسماح بدمج مرن للقدرات، أو حتى إزالة قدرات غير مرغوبة
    (مثل: إزالة ميل للسمية عبر "طرح" متجه مهمة سلبي).
    """
    def __init__(self, base_model_state_dict: dict):
        self.base_state_dict = base_model_state_dict

    def extract_task_vector(self, finetuned_state_dict: dict) -> dict:
        """استخراج متجه المهمة = الفرق بين النموذج المُضبَّط والأساسي."""
        task_vector = {}
        for key in self.base_state_dict.keys():
            task_vector[key] = (
                finetuned_state_dict[key].float() - self.base_state_dict[key].float()
            )
        return task_vector

    def apply_task_vectors(self, task_vectors_with_weights: list) -> dict:
        """
        دمج عدة متجهات مهام بأوزان (λ) مختلفة فوق النموذج الأساسي.
        task_vectors_with_weights: قائمة من (task_vector_dict, lambda_weight)
        """
        merged_state_dict = {}
        for key in self.base_state_dict.keys():
            combined_delta = sum(
                tv[key] * weight for tv, weight in task_vectors_with_weights
            )
            merged_state_dict[key] = (
                self.base_state_dict[key].float() + combined_delta
            ).to(self.base_state_dict[key].dtype)
        return merged_state_dict

    def negate_task_vector(self, task_vector: dict, strength: float = 1.0) -> dict:
        """
        إزالة مهارة/سلوك: طرح متجه المهمة بدل جمعه. مفيد لإزالة
        سلوكيات غير مرغوبة (مثلًا: متجه مهمة مُستخرَج من نموذج
        مُدرَّب عمدًا على استجابات سامة، ثم طرحه لتقليل السمية).
        """
        return self.apply_task_vectors([(task_vector, -strength)])

# --- مثال استخدام: دمج قدرة برمجية وقدرة رياضية في نموذج عام ---
arithmetic = TaskArithmetic(base_model_state_dict=base_llama_weights)

code_task_vector = arithmetic.extract_task_vector(code_finetuned_weights)
math_task_vector = arithmetic.extract_task_vector(math_finetuned_weights)

# دمج القدرتين معًا بأوزان متساوية (λ=0.5 لكل منهما شائع كقيمة ابتدائية)
merged_weights = arithmetic.apply_task_vectors([
    (code_task_vector, 0.5),
    (math_task_vector, 0.5),
])


### 3.3 TIES-Merging وDARE: حل مشكلة تداخل التحديثات

🔴 طبقة التعمّق

المشكلة الأساسية في الجمع البسيط لـ Task Vectors: تداخل الإشارات (Sign Interference) — عندما يُحدِّث نموذجان مُضبَّطان نفس الوزن في اتجاهين متعاكسين (أحدهما يزيده، الآخر يُنقِصه)، فإن الجمع المباشر "يُلغي" التحديثين جزئيًا، مما يُضعِف كلتا القدرتين في النموذج المدموج.

TIES-Merging (TrIm, Elect Sign, merge) يحل هذا بثلاث خطوات:


In [ ]:
class TIESMerging:
    """
    تطبيق كامل لخوارزمية TIES-Merging (Yadav et al., 2023):
    حل مشكلة تداخل الإشارات بين عدة Task Vectors عبر ثلاث خطوات
    منهجية: التقليم (Trim)، انتخاب الإشارة (Elect Sign)، والدمج
    الانتقائي (Disjoint Merge).
    """
    def __init__(self, base_state_dict: dict, trim_density: float = 0.2):
        self.base_state_dict = base_state_dict
        self.trim_density = trim_density  # نسبة القيم المُحتفَظ بها (الأعلى قيمة)

    def trim_task_vector(self, task_vector: dict) -> dict:
        """
        الخطوة 1 - Trim: الاحتفاظ فقط بأعلى k% من القيم المطلقة لكل
        متجه مهمة (القيم الأكبر تأثيرًا)، وتصفير الباقي. هذا يُزيل
        "الضجيج" الصغير الذي يُسبِّب معظم التداخل غير المفيد.
        """
        trimmed = {}
        for key, tensor in task_vector.items():
            flat = tensor.flatten().abs()
            k = int(len(flat) * self.trim_density)
            if k == 0:
                trimmed[key] = torch.zeros_like(tensor)
                continue
            threshold = torch.topk(flat, k).values.min()
            mask = tensor.abs() >= threshold
            trimmed[key] = tensor * mask
        return trimmed

    def elect_sign(self, trimmed_task_vectors: list) -> dict:
        """
        الخطوة 2 - Elect Sign: لكل وزن، تحديد "الإشارة الغالبة" عبر
        كل متجهات المهام (بناءً على مجموع القيم الموجبة مقابل السالبة
        المُرجَّح بالحجم)، ثم الاحتفاظ لاحقًا فقط بالتحديثات المتوافقة
        مع هذه الإشارة الغالبة.
        """
        elected_signs = {}
        for key in self.base_state_dict.keys():
            stacked = torch.stack([tv[key] for tv in trimmed_task_vectors])
            sign_sum = stacked.sum(dim=0)
            elected_signs[key] = torch.sign(sign_sum)
        return elected_signs

    def disjoint_merge(self, trimmed_task_vectors: list,
                        elected_signs: dict) -> dict:
        """
        الخطوة 3 - Disjoint Merge: لكل وزن، نأخذ متوسط القيم فقط من
        متجهات المهام التي "تتفق" إشارتها مع الإشارة الغالبة المُنتخَبة،
        متجاهلين تمامًا القيم المتعارضة (بدل إلغائها بالجمع كما في
        Task Arithmetic البسيط).
        """
        merged_delta = {}
        for key in self.base_state_dict.keys():
            stacked = torch.stack([tv[key] for tv in trimmed_task_vectors])
            sign = elected_signs[key]

            # قناع القيم المتوافقة مع الإشارة الغالبة فقط
            agreement_mask = (torch.sign(stacked) == sign.unsqueeze(0)) & (stacked != 0)
            agreement_count = agreement_mask.sum(dim=0).clamp(min=1)

            masked_values = stacked * agreement_mask
            merged_delta[key] = masked_values.sum(dim=0) / agreement_count

        return merged_delta

    def merge(self, task_vectors: list, scaling_factor: float = 1.0) -> dict:
        """الخط الكامل: Trim → Elect Sign → Disjoint Merge → إضافة للأساس."""
        trimmed_vectors = [self.trim_task_vector(tv) for tv in task_vectors]
        elected_signs = self.elect_sign(trimmed_vectors)
        merged_delta = self.disjoint_merge(trimmed_vectors, elected_signs)

        final_state_dict = {}
        for key in self.base_state_dict.keys():
            final_state_dict[key] = (
                self.base_state_dict[key].float()
                + scaling_factor * merged_delta[key]
            ).to(self.base_state_dict[key].dtype)
        return final_state_dict


DARE (Drop And REscale) تقنية مُكمِّلة بسيطة وفعّالة جدًا:


In [ ]:
def dare_sparsify_task_vector(task_vector: dict, drop_rate: float = 0.9) -> dict:
    """
    DARE (Yu et al., 2023): إسقاط عشوائي (Random Drop) لنسبة كبيرة جدًا
    من قيم متجه المهمة (حتى 90-99%!)، مع إعادة قياس (Rescale) القيم
    المتبقية للتعويض عن الكتلة المفقودة. النتيجة المُدهشة: الأداء
    يبقى شبه ثابت حتى مع drop_rate=0.9، مما يُظهر أن معظم التحديثات
    في Task Vector "زائدة عن الحاجة" (Redundant) ويمكن حذفها بأمان،
    وهذا يُقلِّل تداخل الإشارات بشكل كبير عند الدمج اللاحق.
    """
    sparsified = {}
    for key, tensor in task_vector.items():
        mask = torch.rand_like(tensor) > drop_rate  # الاحتفاظ بنسبة (1-drop_rate)
        rescale_factor = 1.0 / (1.0 - drop_rate)  # إعادة القياس للتعويض
        sparsified[key] = tensor * mask * rescale_factor
    return sparsified

# DARE غالبًا تُستخدم كخطوة تمهيدية قبل TIES أو Task Arithmetic
# (DARE-TIES: تطبيق DARE أولاً لتخفيف الكثافة، ثم TIES للدمج النهائي)


### 3.4 SLERP وFrankenmerging (نهج mergekit)

🟡 طبقة الآلية

SLERP (Spherical Linear Interpolation) بديل أفضل رياضيًا من المتوسط الخطي البسيط (Linear Interpolation) عند دمج نموذجين فقط، لأنه يحترم الطبيعة الهندسية الكروية لفضاء الأوزان عالي الأبعاد بدل افتراض استقامة خطية بينها.


In [ ]:
def slerp_merge(weight_a: torch.Tensor, weight_b: torch.Tensor,
                 t: float = 0.5, epsilon: float = 1e-8) -> torch.Tensor:
    """
    الاستيفاء الخطي الكروي (SLERP) بين وزنين - أكثر دقة هندسيًا من
    المتوسط الخطي البسيط lerp = (1-t)*a + t*b عند دمج نموذجين فقط،
    لأنه يحافظ على "القاعدة" (norm) الهندسية للمتجهات بدل تقليصها
    بشكل مصطنع كما يحدث أحيانًا مع المتوسط الخطي المباشر.
    """
    a_flat = weight_a.flatten().float()
    b_flat = weight_b.flatten().float()

    a_norm = a_flat / (a_flat.norm() + epsilon)
    b_norm = b_flat / (b_flat.norm() + epsilon)

    # زاوية التشابه بين المتجهين
    dot_product = torch.clamp((a_norm * b_norm).sum(), -1.0, 1.0)
    omega = torch.acos(dot_product)
    sin_omega = torch.sin(omega) + epsilon

    # إذا كانت الزاوية صغيرة جدًا (متجهات شبه متماثلة)، استخدم lerp عادي كبديل آمن
    if sin_omega.abs() < 1e-6:
        result_flat = (1 - t) * a_flat + t * b_flat
    else:
        coeff_a = torch.sin((1 - t) * omega) / sin_omega
        coeff_b = torch.sin(t * omega) / sin_omega
        result_flat = coeff_a * a_flat + coeff_b * b_flat

    return result_flat.reshape(weight_a.shape).to(weight_a.dtype)

def slerp_merge_full_models(state_dict_a: dict, state_dict_b: dict,
                             t: float = 0.5) -> dict:
    """تطبيق SLERP على كل أوزان النموذج طبقة بطبقة."""
    merged = {}
    for key in state_dict_a.keys():
        merged[key] = slerp_merge(state_dict_a[key], state_dict_b[key], t=t)
    return merged


Frankenmerging (Passthrough Merging): دمج أكثر جذرية — تركيب طبقات كاملة من نماذج مختلفة فوق بعضها (وليس مجرد مزج الأوزان داخل نفس البنية)، وهو ما يُستخدم أيضًا في Depth Upscaling (الفصل 28).


In [ ]:
def frankenmerge_layers(model_a_layers: list, model_b_layers: list,
                         layer_recipe: list) -> list:
    """
    Frankenmerging: بناء نموذج جديد بتجميع طبقات كاملة من نماذج مختلفة
    (مشابهة البنية) حسب "وصفة" مُحدَّدة يدويًا أو تجريبيًا. هذا النهج
    استخدمته نماذج مثل Goliath-120B (دمج طبقات من نسختين من Llama 2 70B)
    لإنتاج نموذج أكبر وأحيانًا أقوى دون أي تدريب إضافي على الإطلاق.

    layer_recipe: قائمة من tuples (المصدر, رقم_الطبقة)
    مثال: [("a", 0), ("a", 1), ("b", 0), ("a", 2), ("b", 1), ...]
    """
    new_model_layers = []
    for source, layer_idx in layer_recipe:
        if source == "a":
            new_model_layers.append(copy.deepcopy(model_a_layers[layer_idx]))
        elif source == "b":
            new_model_layers.append(copy.deepcopy(model_b_layers[layer_idx]))
    return new_model_layers


> 💡 نصيحة عملية: أداة mergekit (مفتوحة المصدر من Arcee AI) تُطبِّق كل هذه التقنيات (TIES، DARE، SLERP، Frankenmerging، وغيرها) عبر ملف تهيئة YAML بسيط دون كتابة كود — وهي الأداة المعيارية الفعلية المستخدمة في إنتاج معظم النماذج المدموجة على Hugging Face Open LLM Leaderboard.


```yaml
# مثال ملف تهيئة mergekit لدمج TIES بين ثلاثة نماذج مُضبَّطة
merge_method: ties
base_model: meta-llama/Meta-Llama-3-8B
models:
  - model: nmodel-finetuned-on-code
    parameters:
      weight: 0.4
      density: 0.5   # نسبة trim_density
  - model: model-finetuned-on-math
    parameters:
      weight: 0.4
      density: 0.5
  - model: model-finetuned-on-arabic-chat
    parameters:
      weight: 0.2
      density: 0.5
parameters:
  normalize: true
dtype: bfloat16
```


## 4. مثال محلول خطوة بخطوة

نأخذ مثالًا عدديًا صغيرًا يوضّح لماذا لا يكفي الجمع المباشر لمتجهات المهام.


In [ ]:
import torch

# أوزان أساس افتراضية (3 قيم فقط للتوضيح)
theta_base = torch.tensor([1.0, -0.5, 0.2])

# متجه مهمة A (برمجة) ومتجه مهمة B (رياضيات)
tau_a = torch.tensor([0.4, 0.3, -0.1])
tau_b = torch.tensor([0.2, -0.4, 0.3])

# 1) Task Arithmetic البسيط: جمع مباشر
tau_sum = tau_a + tau_b
print("tau_sum       =", tau_sum.tolist())        # [0.6, -0.1, 0.2]
print("theta_sum     =", (theta_base + tau_sum).tolist())

# 2) TIES مبسّط: تقليم القيم الصغيرة ثم انتخاب الإشارة الغالبة
def trim(tau, keep=2):
    threshold = tau.abs().topk(keep).values.min()
    return torch.where(tau.abs() >= threshold, tau, torch.zeros_like(tau))

ta, tb = trim(tau_a), trim(tau_b)
print("trimmed A     =", ta.tolist())             # [0.4, 0.3, 0.0]
print("trimmed B     =", tb.tolist())             # [0.0, -0.4, 0.3]

# الإشارة الغالبة لكل موضع: مجموع القيم الموقّعة
elected = torch.sign(ta + tb)
print("elected sign  =", elected.tolist())        # [1, -1, 1]

# الدمج المنفصل: متوسط القيم المتوافقة فقط مع الإشارة الغالبة
mask_a = (torch.sign(ta) == elected) & (ta != 0)
mask_b = (torch.sign(tb) == elected) & (tb != 0)
tau_ties = torch.where(mask_a, ta, torch.zeros_like(ta)) + \
           torch.where(mask_b, tb, torch.zeros_like(tb))
print("tau_ties      =", tau_ties.tolist())       # [0.4, -0.4, 0.3]
print("theta_ties    =", (theta_base + tau_ties).tolist())


لاحظ الفرق في الموضع الثاني: الجمع المباشر أنتج `-0.1` فقط لأن `+0.3` و`-0.4`
ألغى كل منهما الآخر جزئيًا. أما TIES فانتخب الإشارة السالبة (لأن قوة التحديث
السلبي أكبر) واحتفظ بـ `-0.4` كاملة، فحافظ على اتجاه مهمة الرياضيات بدل
إضعافه. هذا هو جوهر «تداخل الإشارة» وحلّه.


## 5. نصائح وحيل

- **ابدأ بـ TIES + DARE** كوصفة افتراضية عند دمج عدة نماذج لمهام مختلفة من نفس الأساس.
- **استخدم `mergekit`** في الإنتاج بدل كتابة الكود يدويًا؛ فهو يوفّر TIES وDARE وSLERP وFrankenmerging عبر YAML مُختبَر.
- **طابق المعمارية والتوكنizer** تمامًا؛ لا تدمج نموذجين بأحجام طبقات أو مفردات مختلفة.
- **احتفظ دائمًا بالنموذج الأساسي (Base)**؛ كل تقنيات Task Arithmetic تُعرَّف بالنسبة إليه.
- **اضبط `density` و`weight` تجريبيًا** على مجموعة تحقق صغيرة قبل الاعتماد على القيم الافتراضية.
- **قيّم على مصفوفة مهام** (وليس مهمة واحدة) لرصد التدهور غير المتوقع في قدرات أخرى.
- **وثّق الوصفة** (الأوزان، densities، إصدار mergekit) لأن إعادة إنتاج الدمج شرط أساسي للنشر.
- **جرّب Greedy Soup** عندما تكون النماذج نسخًا من نفس المهمة بإعدادات مختلفة، وTIES عندما تختلف المهام.


## 6. أخطاء شائعة ومصائد

دمج نماذج ذات معماريات مختلفة تمامًا (أحجام طبقات مختلفة) مباشرة — الدمج على مستوى الأوزان يتطلب تطابق الشكل (shape) الكامل بين كل الأوزان المُقابلة.
تجاهل التحقق من الأداء على معايير متعددة بعد الدمج — أحيانًا الدمج يُحسِّن معيارًا واحدًا بينما يُضعِف آخر بشكل غير متوقع؛ استخدم مجموعة تقييم شاملة (الجزء التاسع) دائمًا.
استخدام أوزان λ متساوية افتراضيًا لكل النماذج المدموجة دون تجريب — عادة التجريب على عدة قيم λ (أو استخدام Greedy Soup/Evolutionary Merging) يُحسِّن النتيجة كثيرًا مقارنة بالتساوي الافتراضي.


## 7. جدول مقارنة

| التقنية | عدد النماذج المدعوم | يحل مشكلة تداخل الإشارة؟ | التعقيد | الأداء النموذجي |
|---|---|---|---|---|
| Uniform/Greedy Soup | عدة (نفس المعمارية والمهمة تقريبًا) | لا (ضمنيًا عبر التشابه) | بسيط جدًا | جيد عندما تكون النماذج متقاربة جدًا |
| Task Arithmetic (بسيط) | عدة مهام مختلفة | لا | بسيط | متوسط - قد يعاني من التداخل |
| TIES-Merging | عدة مهام مختلفة | نعم (صراحةً) | متوسط | قوي جدًا - الخيار الافتراضي الموصى به |
| DARE | يُضاف فوق أي تقنية أخرى | يُخفِّف التداخل تمهيديًا | بسيط | يُحسِّن أي تقنية أخرى عند الدمج معها |
| SLERP | نموذجان فقط حصرًا | جزئيًا (هندسيًا) | بسيط | ممتاز لدمج نموذجين متقاربين |
| Frankenmerging | عدة (طبقات كاملة) | غير ذي صلة (مفهوم مختلف) | يتطلب تجريب يدوي | متفاوت جدًا - يعتمد على "الوصفة" |


## 8. ملخص أفضل الممارسات

✅ ابدأ دائمًا بـ TIES-Merging كخيار افتراضي قوي عند دمج عدة نماذج مُضبَّطة لمهام مختلفة من نفس الأساس.
✅ استخدم DARE كخطوة تمهيدية (Sparsification) قبل أي تقنية دمج أخرى لتقليل التداخل.
✅ استخدم mergekit بدل كتابة الكود يدويًا في المشاريع الإنتاجية — أكثر موثوقية ومُختبَرة على نطاق واسع.
✅ قيّم دائمًا النموذج المدموج على مجموعة شاملة من المعايير (ليس فقط المعيار الذي استهدفته)، لرصد أي تدهور غير متوقع في قدرات أخرى.


## 9. أسئلة مقابلات

**سؤال: ما هي مشكلة "تداخل الإشارة" (Sign Interference) في دمج النماذج، وكيف تحلها خوارزمية TIES-Merging تحديدًا؟**

**إجابة نموذجية:** عندما نجمع عدة Task Vectors مباشرة (Task Arithmetic البسيط)، قد يحدث أن وزنًا معينًا يحتاج زيادة في مهمة ما (قيمة موجبة في متجه المهمة الأول) بينما يحتاج نفس الوزن نقصانًا في مهمة أخرى (قيمة سالبة في متجه المهمة الثاني) — جمعهما المباشر يُنتج قيمة صغيرة أو شبه صفرية تُلغي كلا التحديثين المفيدين بدل الاستفادة منهما، وهذا يُضعِف أداء النموذج المدموج في كلتا المهمتين. TIES-Merging تحل هذا عبر ثلاث خطوات: (1) Trim - تصفير القيم الصغيرة غير المهمة في كل متجه مهمة للتخلص من الضجيج، (2) Elect Sign - تحديد الإشارة "الغالبة" لكل وزن عبر كل متجهات المهام مجتمعة (بناءً على القوة الإجمالية لكل اتجاه)، (3) Disjoint Merge - حساب المتوسط فقط من القيم المتوافقة مع الإشارة الغالبة المُنتخَبة، وتجاهل القيم المتعارضة كليًا بدل جمعها وإلغائها. هذا يضمن أن التحديث النهائي لكل وزن ينعكس باتجاه الإجماع الحقيقي بين المهام، لا متوسطًا مشوَّهًا بتداخل متعارض.

**سؤال: لماذا تُعتبر SLERP أكثر ملاءمة رياضيًا من المتوسط الخطي البسيط (Linear Interpolation) عند دمج نموذجين؟**

**إجابة نموذجية:** فضاء أوزان الشبكات العصبية عالي الأبعاد، والمتجهات (الأوزان) فيه غالبًا ما تحمل معلومة مهمة في اتجاهها (orientation) بقدر أهمية مقدارها (magnitude/norm). المتوسط الخطي البسيط بين متجهين (a+b)/2 يُنتج متجهًا قد يقع "داخل" الكرة الفاصلة بين المتجهين الأصليين بدل على سطحها، مما يُقلِّص قاعدة (norm) المتجه الناتج بشكل مصطنع إذا كانت الزاوية بين a وb كبيرة — وهذا قد يُفسَد تمثيل الشبكة. SLERP يحسب الاستيفاء على طول القوس الكروي الواصل بين المتجهين (بدل الخط المستقيم)، محافظًا على القاعدة الهندسية للمتجهات طوال عملية الاستيفاء، مما يحافظ على "سلامة" التمثيل الداخلي للنموذج بشكل أفضل عمليًا، خصوصًا عند دمج نموذجين تباعدا بشكل ملحوظ عن بعضهما أثناء الضبط الدقيق المنفصل.


## 10. تمرين عملي

باستخدام mergekit (أو تطبيق TIESMerging المكتوب أعلاه)، ادمج ثلاث نسخ مُضبَّطة دقيقًا (بـ LoRA مثلًا) من نفس نموذج أساسي صغير (TinyLlama أو Qwen 0.5B) على ثلاث مهام مختلفة (تلخيص، أسئلة وأجوبة، تصنيف مشاعر)، وقيّم النموذج المدموج على الثلاث مهام معًا مقارنة بكل نموذج فردي.


## 11. مراجع للتعمّق

- Wortsman et al. (2022): Model Soups: Averaging Weights of Multiple Fine-tuned Models
- Ilharco et al. (2022): Editing Models with Task Arithmetic
- Yadav et al. (2023): TIES-Merging: Resolving Interference When Merging Models
- Yu et al. (2023): Language Models are Super Mario: Absorbing Abilities from Homologous Models (DARE)
- Goddard et al. (2024, Arcee AI): Arcee's MergeKit: A Toolkit for Merging Large Language Models


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — إعادة استخدام النماذج / الفصل 26: دمج النماذج: Model Merging.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
